# CREMP Benchmark — Hybrid ΔPSA Validation

**Goal:** Validate the two-population chameleonicity model using physically-grounded membrane conformers.

**Strategy (Option C):**
- **Aqueous PSA** → vacuum ETKDGv3 max-PSA (existing pipeline, `feature_matrix.csv`)
- **Membrane PSA** → CREMP CHCl₃ min-PSA (pre-computed CREST ensemble at 298.15 K)
- **Hybrid ΔPSA** = vacuum aq-PSA − CREMP mem-PSA

**Test:** Does replacing the vacuum membrane conformer with a CREST CHCl₃ conformer improve recovery of the chameleonic/rigid two-population signal vs CycPeptMPDB PAMPA permeability labels?

**Compounds:** CREMP × CycPeptMPDB overlap (SMILES-matched)

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
# Colab: uncomment below
# !pip install -q rdkit scikit-learn umap-learn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
from rdkit import Chem
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
RESULTS_DIR    = Path('../results')
CREMP_PSA_CSV  = RESULTS_DIR / 'cremp_deltapsa.csv'
FEATURE_MATRIX = RESULTS_DIR / 'feature_matrix.csv'

print('Paths set.')

## 1. Load Data

In [ ]:
# ── CREMP ΔPSA (CHCl₃ conformers) ─────────────────────────────────────────────
cremp = pd.read_csv(CREMP_PSA_CSV, low_memory=False)
cremp = cremp[cremp['error'].isna()].copy()
print(f'CREMP compounds (clean): {len(cremp)}')
print(f'Columns: {list(cremp.columns)}')
cremp.head(3)

In [ ]:
# ── CycPeptMPDB feature matrix ─────────────────────────────────────────────────
fm = pd.read_csv(FEATURE_MATRIX, low_memory=False)
print(f'CycPeptMPDB compounds: {len(fm)}')
print(f'Permeable: {fm["permeable"].sum()} | Impermeable: {(fm["permeable"]==0).sum()}')

fm_keep = ['ID', 'SMILES_canonical', 'permeable', 'PAMPA', 'MolWt',
           'Monomer_Length', 'aq_psa3d', 'mem_psa3d', 'delta_psa3d',
           'psa3d_std', 'psa3d_spread']
fm_keep = [c for c in fm_keep if c in fm.columns]
fm = fm[fm_keep].copy()
fm.head(3)

## 2. SMILES Matching — CREMP × CycPeptMPDB Overlap

In [ ]:
def canonical_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        return Chem.MolToSmiles(mol) if mol else None
    except Exception:
        return None

print('Canonicalizing SMILES...')
cremp['canon_smi'] = cremp['smiles'].apply(canonical_smiles)
fm['canon_smi']    = fm['SMILES_canonical'].apply(canonical_smiles)

print(f'CREMP valid SMILES:       {cremp["canon_smi"].notna().sum()} / {len(cremp)}')
print(f'CycPeptMPDB valid SMILES: {fm["canon_smi"].notna().sum()} / {len(fm)}')

In [ ]:
# ── Inner join on canonical SMILES ────────────────────────────────────────────
merged = pd.merge(
    cremp.dropna(subset=['canon_smi']),
    fm.dropna(subset=['canon_smi']),
    on='canon_smi',
    how='inner',
    suffixes=('_cremp', '_vac')
)

print(f'Overlap compounds: {len(merged)}')
print(f'Permeable: {merged["permeable"].sum()} | Impermeable: {(merged["permeable"]==0).sum()}')

# ── Hybrid ΔPSA ───────────────────────────────────────────────────────────────
# aq-PSA: vacuum ETKDGv3 max-PSA (best extended conformer available)
# mem-PSA: CREMP CHCl₃ CREST min-PSA (physically grounded collapsed conformer)
merged['hybrid_delta_psa'] = merged['aq_psa3d'] - merged['mem_psa3d_cremp']

print(f'\nHybrid ΔPSA: {merged["hybrid_delta_psa"].min():.1f} – {merged["hybrid_delta_psa"].max():.1f} Å²  (mean={merged["hybrid_delta_psa"].mean():.1f})')
print(f'Vacuum ΔPSA: {merged["delta_psa3d"].min():.1f} – {merged["delta_psa3d"].max():.1f} Å²  (mean={merged["delta_psa3d"].mean():.1f})')

## 3. Membrane Conformer: Vacuum vs CREMP CHCl₃

In [ ]:
sub = merged[['mem_psa3d_vac', 'mem_psa3d_cremp', 'permeable']].dropna()
r, p = stats.pearsonr(sub['mem_psa3d_vac'], sub['mem_psa3d_cremp'])
diff = sub['mem_psa3d_cremp'] - sub['mem_psa3d_vac']
mae  = np.abs(diff).mean()
frac_cremp_lower = (diff < 0).mean()

print(f'Pearson r (vacuum vs CREMP mem-PSA): {r:.3f}  (p={p:.2e})')
print(f'MAE:                                  {mae:.1f} Å²')
print(f'CREMP finds lower mem-PSA:            {frac_cremp_lower*100:.1f}% of cases')
print(f'Mean diff (CREMP − vacuum):           {diff.mean():.1f} ± {diff.std():.1f} Å²')
print()
print('Interpretation:')
if diff.mean() < -5:
    print('  CREMP finds systematically more collapsed conformers — vacuum over-estimates mem-PSA.')
elif diff.mean() < -2:
    print('  Modest offset — vacuum is a reasonable but imperfect proxy for collapsed conformer.')
else:
    print('  Vacuum and CREMP mem-PSA largely agree for this compound class.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sc = ax.scatter(sub['mem_psa3d_vac'], sub['mem_psa3d_cremp'],
                c=sub['permeable'], cmap='RdYlGn', alpha=0.6, s=20, linewidths=0)
lo = min(sub['mem_psa3d_vac'].min(), sub['mem_psa3d_cremp'].min()) - 5
hi = max(sub['mem_psa3d_vac'].max(), sub['mem_psa3d_cremp'].max()) + 5
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.5, label='y=x')
ax.set_xlabel('Vacuum mem-PSA (Å²)', fontsize=11)
ax.set_ylabel('CREMP CHCl₃ mem-PSA (Å²)', fontsize=11)
ax.set_title(f'Membrane Conformer PSA\nr={r:.3f}, MAE={mae:.1f} Å²', fontsize=12)
plt.colorbar(sc, ax=ax, label='Permeable')
ax.legend(fontsize=9)

ax = axes[1]
perm_diff   = diff[sub['permeable'] == 1]
imperm_diff = diff[sub['permeable'] == 0]
ax.hist(imperm_diff, bins=30, alpha=0.6, color='tomato',   label=f'Impermeable (n={len(imperm_diff)})', density=True)
ax.hist(perm_diff,   bins=30, alpha=0.6, color='seagreen', label=f'Permeable (n={len(perm_diff)})',     density=True)
ax.axvline(0, color='k', lw=1, ls='--')
ax.set_xlabel('CREMP mem-PSA − Vacuum mem-PSA (Å²)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Δ(mem-PSA) by permeability class', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'cremp_membrane_conformer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Two-Population Test — AUC Comparison

In [ ]:
df_auc = merged[['permeable', 'delta_psa3d', 'hybrid_delta_psa', 'psa3d_spread_cremp']].dropna(subset=['permeable', 'delta_psa3d', 'hybrid_delta_psa'])

auc_results = {}
for col, label in [
    ('delta_psa3d',         'Vacuum ΔPSA (ETKDGv3)'),
    ('hybrid_delta_psa',    'Hybrid ΔPSA (vac aq / CREMP mem)'),
]:
    if col not in df_auc.columns:
        continue
    tmp = df_auc[['permeable', col]].dropna()
    auc = roc_auc_score(tmp['permeable'], tmp[col])
    auc_results[label] = auc
    print(f'{label:45s}  AUC = {auc:.3f}')

if 'psa3d_spread_cremp' in df_auc.columns:
    tmp = df_auc[['permeable', 'psa3d_spread_cremp']].dropna()
    if len(tmp) > 10:
        auc = roc_auc_score(tmp['permeable'], tmp['psa3d_spread_cremp'])
        auc_results['CREMP PSA spread (CHCl₃ only)'] = auc
        print(f'{"CREMP PSA spread (CHCl₃ only)":45s}  AUC = {auc:.3f}')

print(f'\nn = {len(df_auc)} overlap compounds')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for col, label, color, ls in [
    ('delta_psa3d',      'Vacuum ΔPSA',            'steelblue', '--'),
    ('hybrid_delta_psa', 'Hybrid ΔPSA (CREMP mem)', 'seagreen',  '-'),
]:
    tmp = df_auc[['permeable', col]].dropna()
    if len(tmp) < 10:
        continue
    fpr, tpr, _ = roc_curve(tmp['permeable'], tmp[col])
    auc = roc_auc_score(tmp['permeable'], tmp[col])
    ax.plot(fpr, tpr, color=color, ls=ls, lw=2, label=f'{label}  AUC={auc:.3f}')

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random (0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC: Vacuum vs Hybrid ΔPSA\nCREMP × CycPeptMPDB overlap', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'cremp_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Two-Population Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

perm   = merged[merged['permeable'] == 1]
imperm = merged[merged['permeable'] == 0]

for ax, col, title in [
    (axes[0], 'delta_psa3d',      'Vacuum ΔPSA'),
    (axes[1], 'hybrid_delta_psa', 'Hybrid ΔPSA (CREMP mem)'),
]:
    ax.hist(imperm[col].dropna(), bins=40, alpha=0.6, color='tomato',
            density=True, label=f'Impermeable (n={len(imperm[col].dropna())})')
    ax.hist(perm[col].dropna(),   bins=40, alpha=0.6, color='seagreen',
            density=True, label=f'Permeable (n={len(perm[col].dropna())})')
    u, p = stats.mannwhitneyu(perm[col].dropna(), imperm[col].dropna(), alternative='greater')
    ax.set_xlabel('ΔPSA (Å²)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{title}\nMann-Whitney p={p:.3e}', fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle('Two-Population Test: Chameleonic vs Rigid', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'cremp_two_population.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary

In [ ]:
print('=' * 65)
print('CREMP BENCHMARK SUMMARY')
print('=' * 65)
print(f'CREMP compounds processed:       {len(cremp)}')
print(f'CycPeptMPDB compounds:           {len(fm)}')
print(f'Overlap (SMILES-matched):        {len(merged)}')
print(f'  Permeable:                     {merged["permeable"].sum()}')
print(f'  Impermeable:                   {(merged["permeable"]==0).sum()}')
print()
print('Membrane conformer quality (CHCl₃ vs vacuum):')
print(f'  Pearson r:                     {r:.3f}')
print(f'  MAE:                           {mae:.1f} Å²')
print(f'  CREMP finds lower mem-PSA:     {frac_cremp_lower*100:.1f}% of cases')
print()
print('AUC (PAMPA permeability):')
for label, auc in auc_results.items():
    print(f'  {label:45s} {auc:.3f}')
print()
print('Interpretation:')
if 'Hybrid ΔPSA (vac aq / CREMP mem)' in auc_results and 'Vacuum ΔPSA (ETKDGv3)' in auc_results:
    delta_auc = auc_results['Hybrid ΔPSA (vac aq / CREMP mem)'] - auc_results['Vacuum ΔPSA (ETKDGv3)']
    if delta_auc > 0.01:
        print(f'  CREMP mem-PSA improves AUC by +{delta_auc:.3f}')
        print('  Physically-grounded membrane conformer recovers better signal.')
    elif delta_auc < -0.01:
        print(f'  Vacuum mem-PSA outperforms CREMP by {-delta_auc:.3f}.')
        print('  CHCl₃ ensemble may over-collapse rigid compounds too.')
    else:
        print(f'  AUC difference negligible ({delta_auc:+.3f}).')
        print('  PAMPA label noise dominates both signals — conformer quality is not the bottleneck.')